In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df=pd.read_csv("/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv")


# milestone-1 
#### Q1 Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [ ]:
x=df['answer'].value_counts().max() +  df['answer'].value_counts().min()

print("Sum of Most frequent and least frequent options in answer column =",x)

### Q2 After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?   

In [ ]:
import string
has_punctuation = df["prompt"].apply(
    lambda text: any(c in string.punctuation for c in text)
)
print(has_punctuation.unique())

In [ ]:
import string
def clean_prompt(text):
    text=text.lower()
    translator=str.maketrans('','',string.punctuation) #nothing,nothing,delete punctuation 
    text=text.translate(translator)
    return text
    

In [ ]:
df["prompt"]=df["prompt"].apply(clean_prompt)
#verifying 
has_punctuation = df["prompt"].apply(
    lambda text: any(c in string.punctuation for c in text)
)
print(has_punctuation.unique())

In [ ]:
vocab=set()
for text in df["prompt"]:
    vocab.update(text.split())

print(len(vocab))

### Q3 Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string

temp=df[df["id"]==1]["prompt"].iloc[0].split()
filtered_words=[i for i in temp if i not in ENGLISH_STOP_WORDS] 
print("unfiltered string :",temp)
print("filtered string list : ",filtered_words)
print("length of list ",len(filtered_words))

### Q4 Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
combinelist=pd.concat(
    [df["prompt"],
    df["A"],
    df["B"],
     df["C"],
     df["D"],
     df["E"]
    ])
vectorizer=TfidfVectorizer(stop_words="english")
vectorizer.fit(combinelist)

print(len(vectorizer.vocabulary_))


### Q5 Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
row=df[df["id"]==1].iloc[0]
prompt=row["prompt"]
optA=row["A"]
prompt_vec=vectorizer.transform([prompt])
option_vec=vectorizer.transform([optA])
print(f"Similarity score : {cosine_similarity(prompt_vec,option_vec)[0,0]:.4f}")

### Q6 For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.  

In [ ]:
def transform_columns(df, vectorizer):
    return (
        vectorizer.transform(df['prompt']),
        vectorizer.transform(df['A']),
        vectorizer.transform(df['B']),
        vectorizer.transform(df['C']),
        vectorizer.transform(df['D']),
        vectorizer.transform(df['E'])
    )

    
prompt_vec, A_vec, B_vec, C_vec, D_vec, E_vec = transform_columns(df, vectorizer)

prompt_vec.shape #where rows,no_of_unnique_words

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

scores = {
    'A': cosine_similarity(prompt_vec, A_vec).diagonal(),
    'B': cosine_similarity(prompt_vec, B_vec).diagonal(),
    'C': cosine_similarity(prompt_vec, C_vec).diagonal(),
    'D': cosine_similarity(prompt_vec, D_vec).diagonal(),
    'E': cosine_similarity(prompt_vec, E_vec).diagonal()
}

correct_pred=0
for i in range(len(scores["A"])):
    row_scores={
        'A': scores['A'][i],
        'B': scores['B'][i],
        'C': scores['C'][i],
        'D': scores['D'][i],
        'E': scores['E'][i]  
    }
    best_opt='A'
    for option in ['B','C','D','E']:
        if row_scores[option] > row_scores[best_opt]:
            best_opt=option

    if (best_opt == df.loc[i,'answer']):
        correct_pred+=1

accuracy=(correct_pred/df.shape[0])*100
print(f"Accuracy of cosimilarity : {accuracy}")

### Q9 The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [ ]:
def map3(y_true, top3_preds):
    score = 0

    for actual, pred in zip(y_true, top3_preds):
        if actual == pred[0]:
            score += 1.0

        elif actual == pred[1]:
            score += 0.5

        elif actual == pred[2]:
            score += 1/3

    return score / len(y_true)

In [ ]:
counts = df['answer'].value_counts()

top3 = counts.index[:3].tolist()

print(top3)

In [ ]:
y_pred = [top3] * len(df)

In [ ]:
print("Majority class baseline : ",round(map3(df["answer"],y_pred),3))

In [ ]:
y_pred=[]
for i in range(len(scores["A"])):
    row_scores={
        'A': scores['A'][i],
        'B': scores['B'][i],
        'C': scores['C'][i],
        'D': scores['D'][i],
        'E': scores['E'][i]  
    }
    top3=[]
   
    for k in range(3):
        best_opt=None
        best_score=-1
        for option,score in row_scores.items():
            if score > best_score:
                best_score=score
                best_opt=option
        top3.append(best_opt)
        row_scores.pop(best_opt)
    
    y_pred.append(top3)    
print(len(y_pred))

In [ ]:
print(round(map3(df['answer'], y_pred),2))

# Milestone 2 
## Q1 Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.


In [ ]:
from datasets import load_dataset 
dataset=load_dataset("csv",data_files="/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv")
train=dataset["train"]

train=train.map(
    lambda x : {
        "combined_text":x["prompt"]+""+x["A"]
    }
)

# Print first row

# Length of row 51
print(len(train[51]["combined_text"]))
print(type(list(train["prompt"])))


## Q2 Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.vocab_size)


## Q3 Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [ ]:
print("SEP token ",tokenizer.sep_token,tokenizer.sep_token_id)

## Q4 Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [ ]:
encodings=tokenizer(list(train["prompt"]),padding="max_length",truncation=True,max_length=128,return_tensors="pt")
print(encodings["input_ids"].shape)

## Q5  A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [ ]:
from transformers import BertModel
from transformers import logging
logging.set_verbosity_error()


model = BertModel.from_pretrained("bert-base-uncased")

print(model.config.hidden_size)
print(model.config.num_attention_heads)
print(model.config.hidden_size // model.config.num_attention_heads)

## Q6 Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here

In [ ]:
from transformers import  AutoModel
model = AutoModel.from_pretrained("bert-base-uncased")
text = train[0]["prompt"]
print(text)

In [ ]:
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

  ## Q7 Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [ ]:
cls_embedding = outputs.last_hidden_state[0, 0]
print(cls_embedding[:5] )

In [ ]:
answer = cls_embedding[:5].sum().item()
print(round(answer,4))


## Q8 Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places). 

In [ ]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

# Use your existing tokenizer
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# Tokens (to find the index of "fusion")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

# Last layer, first head
last_attention = outputs.attentions[-1]
head0 = last_attention[0, 0]

fusion_index = tokens.index("fusion")

weight = head0[0, fusion_index].item()

print("Attention weight:", round(weight, 4))

## Q9 Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get row 0
prompt = train[0]["prompt"]
option_b = train[0]["B"]


# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(similarity)
print(round(similarity.item(), 4))

## Q10 Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Load data


letters = ["A","B","C","D","E"]

def map3(actual, predicted):
    score = 0

    for a, p in zip(actual, predicted):
        if a == p[0]:
            score += 1
        elif a == p[1]:
            score += 1/2
        elif a == p[2]:
            score += 1/3

    return score / len(actual)


# TF-IDF PIPELINE


corpus = []

for row in train:
    corpus.append(row["prompt"])
    for l in letters:
        corpus.append(row[l])

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(corpus)

tfidf_predictions = []

for row in train:

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = []

    for l in letters:
        option_vec = vectorizer.transform([row[l]])
        score = (prompt_vec @ option_vec.T).toarray()[0][0]
        scores.append(score)

    order = np.argsort(scores)[::-1]
    tfidf_predictions.append([letters[i] for i in order[:3]])

     

In [ ]:

###################################################
# MiniLM PIPELINE
###################################################

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",device="cuda")

minilm_predictions = []

for row in train:

    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    scores = []

    for l in letters:

        option_embedding = model.encode(
            row[l],
            convert_to_tensor=True
        )

        score = util.cos_sim(
            prompt_embedding,
            option_embedding
        ).item()
        scores.append(score)

    order = np.argsort(scores)[::-1]
    minilm_predictions.append([letters[i] for i in order[:3]])



In [ ]:

# RESULTS
answers = train["answer"]

map3_minilm = map3(answers, minilm_predictions)

count = 0

for ans, tfidf_top3, minilm_top3 in zip(
    answers,
    tfidf_predictions,
    minilm_predictions
):
    if ans not in tfidf_top3 and ans in minilm_top3:
        count += 1

print("MiniLM MAP@3 =", round(map3_minilm,6))
print("Count =", count)

In [ ]:
from transformers import pipeline

# Initialize pipeline (defaults to facebook/bart-large-mnli)
classifier = pipeline("zero-shot-classification")

# Prompt from row index 1
sequence = train[1]["prompt"]

# Candidate labels = Options A, B, C
candidate_labels = [
    train[1]["A"],
    train[1]["B"],
    train[1]["C"]
]

# Classify
result = classifier(sequence, candidate_labels)

# View full result
print(result)

# Top-ranked probability
print(round(result["scores"][0], 4))

In [ ]:
from transformers import pipeline

# Initialize pipeline (defaults to facebook/bart-large-mnli)
classifier = pipeline("zero-shot-classification")

# Prompt from row index 1
sequence = train[1]["prompt"]

# Candidate labels = Options A, B, C
candidate_labels = [
    train[1]["A"],
    train[1]["B"],
    train[1]["C"]
]

# Classify
result = classifier(sequence, candidate_labels)

# View full result
print(result)

# Top-ranked probability
print(round(result["scores"][0], 4))

In [ ]:

# Softmax (default)
result_softmax = classifier(
    sequence,
    candidate_labels
)

# Independent sigmoid probabilities
result_sigmoid = classifier(
    sequence,
    candidate_labels,
    multi_label=True
)

softmax_sum = sum(result_softmax["scores"])
sigmoid_sum = sum(result_sigmoid["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax scores:", result_softmax["scores"])
print("Sigmoid scores:", result_sigmoid["scores"])

print("Softmax sum:", softmax_sum)
print("Sigmoid sum:", sigmoid_sum)
print("Answer:", round(difference, 4))

# **Milestone 4**


In [ ]:
import pandas as pd
df4=pd.read_csv("/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv")

Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [ ]:
labels= sorted(df4["answer"].unique())
print(labels)

In [ ]:
label_to_num={ label:i for i,label in enumerate(labels)}
df4["answer"]=df4["answer"].map(label_to_num)
df4["answer"].iloc[150]

Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [ ]:
result=df4["prompt"].iloc[0] +" [SEP] "+ str(df4["B"].iloc[0])
print(len(result))

Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [ ]:
from transformers import AutoTokenizer
import torch 
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# Get row 0
row = df4.iloc[0]
choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

print("input_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)

# Print the second dimension
print("Second dimension:", input_ids.shape[1])

Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?



In [ ]:
tokenize=AutoTokenizer.from_pretrained("bert-base-uncased")
batch = 16
choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]
batch_input_ids=[]
for i in range(batch):
    row=df4.iloc[i]
    encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
    )
    input_ids=encoding["input_ids"]
    batch_input_ids.append(input_ids)

input_ids = torch.stack(batch_input_ids)
print(input_ids.shape)
print("Total token positions:", input_ids.numel())

Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [ ]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = df4.iloc[0]
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
input_ids=encoding["input_ids"].unsqueeze(0)
print(input_ids.shape)
attention_mask = encoding["attention_mask"].unsqueeze(0)
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

print(outputs.logits)
print(outputs.logits.shape)

Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [ ]:


row = df4.iloc[0]
print(row["answer"])
labels = torch.tensor([row["answer"]])
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask,labels=labels)
print(outputs.loss)
print(outputs.loss.shape)
print(outputs.loss.dim())

Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable)

In [ ]:
from datasets import Dataset
df4["answer"].unique()
df100 = df4.iloc[:100].copy()
def preprocess(dataframe):
    choices = [
        f"{dataframe['prompt']} [SEP] {dataframe['A']}",
        f"{dataframe['prompt']} [SEP] {dataframe['B']}",
        f"{dataframe['prompt']} [SEP] {dataframe['C']}",
        f"{dataframe['prompt']} [SEP] {dataframe['D']}",
        f"{dataframe['prompt']} [SEP] {dataframe['E']}",
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": dataframe["answer"]
    }



dataset = Dataset.from_pandas(df100)
dataset = dataset.map(preprocess)
print(dataset[0]["input_ids"])
print("Number of choices:", len(dataset[0]["input_ids"]))
print("Tokens per choice:", len(dataset[0]["input_ids"][0]))
print("Label:", dataset[0]["labels"])

Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    DataCollatorForMultipleChoice,
)


model = get_peft_model(model, config)
df32 = df4.iloc[:32].copy()
dataset = Dataset.from_pandas(df32)
dataset = dataset.map(preprocess)

data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
)

# ----------------------------
# Trainer
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# ----------------------------
# Train
# ----------------------------
trainer.train()

print("Final global_step:", trainer.state.global_step)

In [ ]:
logits = outputs.logits

probs = torch.softmax(logits, dim=1)

print("Logits:", logits)
print("Probabilities:", round(probs[0,4].item(), 4))